# Topic 31 — Overfitting in Deep Learning
### Theory → deliberately overfit → dropout → weight decay → early stopping → batch norm → compare.

Same overfitting concept from Topics 5 and 30, now with the specific tools deep learning uses to
fight it. We'll deliberately create an overfitting model (small dataset, big network, many epochs,
no regularization), then apply each fix and watch the train/val gap shrink.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

digits = load_digits()
X, y = digits.data, digits.target
X_scaled = StandardScaler().fit_transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# Use only a SMALL slice of the data on purpose -- small data + big model = easy overfitting
small_dataset = TensorDataset(X_tensor[:200], y_tensor[:200])
n_train = 140
train_ds, val_ds = random_split(small_dataset, [n_train, 60], generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

## 1. A helper training function (reused for every variant below)

In [ ]:
def train_model(model, train_loader, val_loader, n_epochs=80, lr=0.001, weight_decay=0.0):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    def accuracy(loader):
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for Xb, yb in loader:
                Xb, yb = Xb.to(device), yb.to(device)
                correct += (model(Xb).argmax(1) == yb).sum().item()
                total += yb.size(0)
        return correct / total

    for epoch in range(n_epochs):
        model.train()
        train_losses = []
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                val_losses.append(criterion(model(Xb), yb).item())

        history["train_loss"].append(np.mean(train_losses))
        history["val_loss"].append(np.mean(val_losses))
        history["train_acc"].append(accuracy(train_loader))
        history["val_acc"].append(accuracy(val_loader))
    return history

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].plot(history["train_loss"], label="train"); axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title(f"{title} — Loss"); axes[0].legend()
    axes[1].plot(history["train_acc"], label="train"); axes[1].plot(history["val_acc"], label="val")
    axes[1].set_title(f"{title} — Accuracy"); axes[1].legend()
    plt.tight_layout(); plt.show()

## 2. Deliberately overfit: a big network, small data, no regularization

In [ ]:
class BigNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )
    def forward(self, x): return self.net(x)

overfit_model = BigNet().to(device)
overfit_history = train_model(overfit_model, train_loader, val_loader, n_epochs=80)
plot_history(overfit_history, "Overfit baseline (big net, no regularization)")
print(f"final gap (train_acc - val_acc): {overfit_history['train_acc'][-1] - overfit_history['val_acc'][-1]:.3f}")
# Watch: train accuracy climbs toward 1.0 while val loss eventually starts RISING again --
# the model is memorizing the 140 training examples instead of learning generalizable patterns.

## 3. Fix 1: Dropout

During training, dropout randomly "turns off" a fraction of neurons on each forward pass, forcing
the network to not over-rely on any single neuron — a form of built-in ensembling. Disabled
automatically during `.eval()` (this is exactly why we call `.eval()` before validation/inference).

In [ ]:
class DropoutNet(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(), nn.Dropout(p),
            nn.Linear(256, 256), nn.ReLU(), nn.Dropout(p),
            nn.Linear(256, 10),
        )
    def forward(self, x): return self.net(x)

dropout_model = DropoutNet(p=0.5).to(device)
dropout_history = train_model(dropout_model, train_loader, val_loader, n_epochs=80)
plot_history(dropout_history, "With Dropout(p=0.5)")
print(f"final gap: {dropout_history['train_acc'][-1] - dropout_history['val_acc'][-1]:.3f}")

## 4. Fix 2: Weight decay (L2 regularization)

Adds a penalty to the loss proportional to the SIZE of the weights, discouraging any single weight
from growing very large (large weights are often a sign of memorizing noise). In PyTorch, this is
just the `weight_decay` argument to the optimizer — you've already seen the math in Topic 28's AdamW.

In [ ]:
weight_decay_model = BigNet().to(device)
wd_history = train_model(weight_decay_model, train_loader, val_loader, n_epochs=80, weight_decay=0.01)
plot_history(wd_history, "With weight_decay=0.01")
print(f"final gap: {wd_history['train_acc'][-1] - wd_history['val_acc'][-1]:.3f}")

## 5. Fix 3: Early stopping

Stop training as soon as validation loss stops improving, instead of training for a fixed number
of epochs — directly prevents the "train accuracy keeps climbing while val gets worse" pattern
seen in the overfit baseline.

In [ ]:
def train_with_early_stopping(model, train_loader, val_loader, max_epochs=200, patience=10, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(max_epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                val_losses.append(criterion(model(Xb), yb).item())
        val_loss = np.mean(val_losses)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}  # save best weights
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"stopped early at epoch {epoch+1} (no improvement for {patience} epochs)")
            model.load_state_dict(best_state)   # restore the BEST version, not the final (overfit) one
            break

    return history

es_model = BigNet().to(device)
es_history = train_with_early_stopping(es_model, train_loader, val_loader, max_epochs=200, patience=10)

plt.figure(figsize=(6, 4))
plt.plot(es_history["val_loss"])
plt.xlabel("epoch"); plt.ylabel("val loss")
plt.title("Early stopping: training halts once val loss stops improving")
plt.show()

## 6. Fix 4: Batch normalization

Normalizes each layer's inputs (mean 0, variance 1) using statistics computed per mini-batch,
which stabilizes and speeds up training, and provides a mild regularization side-effect
(the per-batch statistics add a small amount of noise).

In [ ]:
class BatchNormNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 10),
        )
    def forward(self, x): return self.net(x)

bn_model = BatchNormNet().to(device)
bn_history = train_model(bn_model, train_loader, val_loader, n_epochs=80)
plot_history(bn_history, "With BatchNorm")
print(f"final gap: {bn_history['train_acc'][-1] - bn_history['val_acc'][-1]:.3f}")

## 7. Data augmentation (brief note — full treatment in Topic 32, image-specific)

For images, augmentation applies random transformations (rotate, flip, crop, adjust brightness) to
training images each epoch, effectively creating "new" training examples on the fly and reducing
overfitting. For text, similar ideas exist (synonym replacement, random deletion) but are used more
cautiously since they can change meaning. We'll see concrete image augmentation code in Topic 32.

## 8. Comparing all approaches' final train/val gap

In [ ]:
comparison = {
    "overfit baseline": overfit_history,
    "dropout": dropout_history,
    "weight_decay": wd_history,
    "batch_norm": bn_history,
}

print(f"{'approach':<20}{'train_acc':<12}{'val_acc':<12}{'gap':<10}")
for name, hist in comparison.items():
    gap = hist["train_acc"][-1] - hist["val_acc"][-1]
    print(f"{name:<20}{hist['train_acc'][-1]:<12.3f}{hist['val_acc'][-1]:<12.3f}{gap:<10.3f}")
# A smaller gap = less overfitting. Regularized versions should show noticeably smaller gaps
# than the baseline, even if raw val accuracy differences are modest on this tiny dataset.

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Try Dropout with p=0.2 and p=0.8 -- how does the train/val gap change at each extreme?
# 2. Try weight_decay=0.1 (much stronger) -- does it start hurting TRAIN accuracy too much
#    (a sign of underfitting from over-regularizing)?
# 3. Combine Dropout AND weight_decay AND BatchNorm in one model -- does the gap shrink further,
#    or do you hit diminishing returns?
# 4. Rerun with the FULL digits dataset (not just 200 samples) -- does overfitting become less
#    severe just from having more data, even with the same BigNet architecture and no regularization?

---
### Next up: **Topic 32 — CNNs** (image-focused; convolution, pooling, and image data augmentation).

Say "next" when you're ready.